# Advanced Python 3.9 Problems with Solutions

## Tutorial Edition

This notebook develops a new set of advanced problems around four Python 3.9 features:

- the `zoneinfo` module,
- multi-argument `math.gcd()` and the new `math.lcm()`,
- dictionary union operators (`|` and `|=`),
- `str.removeprefix()` and `str.removesuffix()`.

The material is intentionally presented as a guided tutorial. Each larger problem is broken into small logical steps, with explanations between the code cells.

The examples are designed to be deterministic and testable.

We will use fixed dates instead of the current clock, and each completed solution includes assertions or consistency checks.

The notebook uses only the Python standard library. On Windows, named IANA time zones usually require the first-party `tzdata` package.

## How to use this notebook

For each problem:

1. Read the scenario and constraints.
2. Predict the result before running the next cell.
3. Use the commented practice cell if you want to attempt the problem independently.
4. Compare your approach with the guided solution.
5. Run the verification cell.

The solutions favor explicit data flow, small functions, clear names, and deterministic tests.

## Environment setup

We begin with the imports used throughout the notebook.

In [1]:
from __future__ import annotations

from datetime import date, datetime, time, timedelta, timezone
from fractions import Fraction
from functools import reduce
from math import gcd, lcm
from pathlib import PurePath
from typing import Any, Iterable, Mapping
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

### Time-zone database preflight

`zoneinfo` is part of Python 3.9 and later, but Windows normally does not ship the IANA time-zone database.

If the following cell fails, run this in a separate notebook cell:

```python
%pip install --upgrade tzdata
```

Then restart the kernel and run the notebook again.

In [2]:
try:
    _preflight_zone = ZoneInfo("Europe/Dublin")
except ZoneInfoNotFoundError as exc:
    raise RuntimeError(
        "Named time zones are unavailable. On Windows, run "
        "`%pip install --upgrade tzdata`, restart the kernel, and rerun."
    ) from exc

print("Time-zone database is available:", _preflight_zone)

Time-zone database is available: Europe/Dublin


We will use a small formatting helper so that aware datetimes are easy to compare.

In [3]:
def describe_datetime(value: datetime) -> str:
    if value.tzinfo is None:
        return f"{value.isoformat()} [naive]"
    return (
        f"{value.isoformat()} | "
        f"zone={value.tzname()} | "
        f"UTC={value.astimezone(timezone.utc).isoformat()}"
    )

# Part 1 — Advanced `zoneinfo` Problems

A reliable datetime workflow normally follows three rules:

1. Know whether a datetime is naive or aware.
2. Represent instants in UTC at system boundaries.
3. Convert to a named local zone only for business rules or presentation.

The problems below make those rules concrete.

## Problem 1 — Sort international events by the actual instant

A conference system receives local event times from three offices:

- Sofia,
- New York,
- Tokyo.

Each office sends a wall-clock time and an IANA zone name. We need to sort the events by the instant at which they actually occur.

A local clock value alone is not enough.

For example, `09:00` in Tokyo and `09:00` in New York are not simultaneous. The zone must be attached before events can be compared meaningfully.

In [4]:
raw_events = [
    {
        "name": "Sofia architecture review",
        "local_time": datetime(2025, 2, 12, 16, 30),
        "zone": "Europe/Sofia",
    },
    {
        "name": "New York product review",
        "local_time": datetime(2025, 2, 12, 9, 0),
        "zone": "America/New_York",
    },
    {
        "name": "Tokyo release review",
        "local_time": datetime(2025, 2, 12, 23, 15),
        "zone": "Asia/Tokyo",
    },
]

raw_events

[{'name': 'Sofia architecture review',
  'local_time': datetime.datetime(2025, 2, 12, 16, 30),
  'zone': 'Europe/Sofia'},
 {'name': 'New York product review',
  'local_time': datetime.datetime(2025, 2, 12, 9, 0),
  'zone': 'America/New_York'},
 {'name': 'Tokyo release review',
  'local_time': datetime.datetime(2025, 2, 12, 23, 15),
  'zone': 'Asia/Tokyo'}]

### Step 1 — Convert one local value manually

The input datetime is naive because it has no `tzinfo`.

Since the accompanying metadata tells us the intended zone, we may attach that zone with `replace(tzinfo=...)`.

In [5]:
sample = raw_events[0]
sample_aware = sample["local_time"].replace(
    tzinfo=ZoneInfo(sample["zone"])
)

print(describe_datetime(sample_aware))

2025-02-12T16:30:00+02:00 | zone=EET | UTC=2025-02-12T14:30:00+00:00


### Step 2 — Normalize every event to UTC

UTC gives us one common timeline. Once every event is represented as an aware datetime, Python can sort them correctly.

In [6]:
def normalize_event(event: Mapping[str, Any]) -> dict[str, Any]:
    local_aware = event["local_time"].replace(
        tzinfo=ZoneInfo(event["zone"])
    )
    return {
        **event,
        "local_aware": local_aware,
        "utc_time": local_aware.astimezone(timezone.utc),
    }


normalized_events = [normalize_event(event) for event in raw_events]
normalized_events

[{'name': 'Sofia architecture review',
  'local_time': datetime.datetime(2025, 2, 12, 16, 30),
  'zone': 'Europe/Sofia',
  'local_aware': datetime.datetime(2025, 2, 12, 16, 30, tzinfo=zoneinfo.ZoneInfo(key='Europe/Sofia')),
  'utc_time': datetime.datetime(2025, 2, 12, 14, 30, tzinfo=datetime.timezone.utc)},
 {'name': 'New York product review',
  'local_time': datetime.datetime(2025, 2, 12, 9, 0),
  'zone': 'America/New_York',
  'local_aware': datetime.datetime(2025, 2, 12, 9, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')),
  'utc_time': datetime.datetime(2025, 2, 12, 14, 0, tzinfo=datetime.timezone.utc)},
 {'name': 'Tokyo release review',
  'local_time': datetime.datetime(2025, 2, 12, 23, 15),
  'zone': 'Asia/Tokyo',
  'local_aware': datetime.datetime(2025, 2, 12, 23, 15, tzinfo=zoneinfo.ZoneInfo(key='Asia/Tokyo')),
  'utc_time': datetime.datetime(2025, 2, 12, 14, 15, tzinfo=datetime.timezone.utc)}]

### Step 3 — Sort by the UTC value

In [7]:
sorted_events = sorted(
    normalized_events,
    key=lambda event: event["utc_time"],
)

for event in sorted_events:
    print(
        f"{event['utc_time'].isoformat()}  "
        f"{event['name']}  "
        f"({event['local_aware'].isoformat()})"
    )

2025-02-12T14:00:00+00:00  New York product review  (2025-02-12T09:00:00-05:00)
2025-02-12T14:15:00+00:00  Tokyo release review  (2025-02-12T23:15:00+09:00)
2025-02-12T14:30:00+00:00  Sofia architecture review  (2025-02-12T16:30:00+02:00)


### Solution verification

The three events should be ordered by actual time, not alphabetically and not by their local clock strings.

In [8]:
assert [event["name"] for event in sorted_events] == [
    "New York product review",
    "Tokyo release review",
    "Sofia architecture review",
]

assert all(
    left["utc_time"] <= right["utc_time"]
    for left, right in zip(sorted_events, sorted_events[1:])
)

print("Problem 1 checks passed.")

Problem 1 checks passed.


### Key idea

Attach the correct named zone first, then compare or store the UTC representation.

Do not sort naive local datetimes from different regions.

## Problem 2 — Observe daylight-saving changes in a fixed UTC schedule

A service runs every Monday at `14:00 UTC`.

The UTC schedule never changes, but the displayed local time can change when a region enters or leaves daylight saving time.

We will examine several Mondays around the 2025 spring transitions in New York and Berlin.

New York and Berlin do not switch on the same date, so there is a period during which the time difference between them is unusual.

In [9]:
utc_run_times = [
    datetime(2025, 3, 3, 14, 0, tzinfo=timezone.utc),
    datetime(2025, 3, 10, 14, 0, tzinfo=timezone.utc),
    datetime(2025, 3, 24, 14, 0, tzinfo=timezone.utc),
    datetime(2025, 3, 31, 14, 0, tzinfo=timezone.utc),
]

new_york = ZoneInfo("America/New_York")
berlin = ZoneInfo("Europe/Berlin")

### Step 1 — Convert each UTC instant to both local zones

In [10]:
schedule_rows = []

for run_time in utc_run_times:
    schedule_rows.append(
        {
            "utc": run_time,
            "new_york": run_time.astimezone(new_york),
            "berlin": run_time.astimezone(berlin),
        }
    )

for row in schedule_rows:
    print(
        row["utc"].date(),
        "| New York:", row["new_york"].strftime("%H:%M %Z"),
        "| Berlin:", row["berlin"].strftime("%H:%M %Z"),
    )

2025-03-03 | New York: 09:00 EST | Berlin: 15:00 CET
2025-03-10 | New York: 10:00 EDT | Berlin: 15:00 CET
2025-03-24 | New York: 10:00 EDT | Berlin: 15:00 CET
2025-03-31 | New York: 10:00 EDT | Berlin: 16:00 CEST


### Step 2 — Measure the local UTC offsets

A change in local display time is caused by a change in the zone's UTC offset.

In [11]:
for row in schedule_rows:
    print(
        row["utc"].date(),
        "| New York offset:", row["new_york"].utcoffset(),
        "| Berlin offset:", row["berlin"].utcoffset(),
    )

2025-03-03 | New York offset: -1 day, 19:00:00 | Berlin offset: 1:00:00
2025-03-10 | New York offset: -1 day, 20:00:00 | Berlin offset: 1:00:00
2025-03-24 | New York offset: -1 day, 20:00:00 | Berlin offset: 1:00:00
2025-03-31 | New York offset: -1 day, 20:00:00 | Berlin offset: 2:00:00


### Step 3 — Turn the operation into a reusable function

In [12]:
def localize_utc_schedule(
    instants: Iterable[datetime],
    zone_names: Iterable[str],
) -> list[dict[str, datetime]]:
    zones = {
        zone_name: ZoneInfo(zone_name)
        for zone_name in zone_names
    }

    result = []
    for instant in instants:
        if instant.tzinfo is None:
            raise ValueError("Every schedule instant must be timezone-aware.")

        utc_instant = instant.astimezone(timezone.utc)
        row = {"UTC": utc_instant}

        for zone_name, zone in zones.items():
            row[zone_name] = utc_instant.astimezone(zone)

        result.append(row)

    return result

In [13]:
localized = localize_utc_schedule(
    utc_run_times,
    ["America/New_York", "Europe/Berlin", "Asia/Tokyo"],
)

for row in localized:
    print({
        key: value.strftime("%Y-%m-%d %H:%M %Z")
        for key, value in row.items()
    })

{'UTC': '2025-03-03 14:00 UTC', 'America/New_York': '2025-03-03 09:00 EST', 'Europe/Berlin': '2025-03-03 15:00 CET', 'Asia/Tokyo': '2025-03-03 23:00 JST'}
{'UTC': '2025-03-10 14:00 UTC', 'America/New_York': '2025-03-10 10:00 EDT', 'Europe/Berlin': '2025-03-10 15:00 CET', 'Asia/Tokyo': '2025-03-10 23:00 JST'}
{'UTC': '2025-03-24 14:00 UTC', 'America/New_York': '2025-03-24 10:00 EDT', 'Europe/Berlin': '2025-03-24 15:00 CET', 'Asia/Tokyo': '2025-03-24 23:00 JST'}
{'UTC': '2025-03-31 14:00 UTC', 'America/New_York': '2025-03-31 10:00 EDT', 'Europe/Berlin': '2025-03-31 16:00 CEST', 'Asia/Tokyo': '2025-03-31 23:00 JST'}


### Solution verification

In [14]:
assert schedule_rows[0]["new_york"].hour == 9
assert schedule_rows[1]["new_york"].hour == 10
assert schedule_rows[2]["berlin"].hour == 15
assert schedule_rows[3]["berlin"].hour == 16

print("Problem 2 checks passed.")

Problem 2 checks passed.


### Key idea

A fixed UTC instant is stable.

A local wall-clock representation may shift because named time zones contain historical and daylight-saving rules.

## Problem 3 — Resolve an ambiguous local time with `fold`

When daylight saving time ends, some local clock values occur twice.

In New York on 3 November 2024, the clock moved backward from daylight time to standard time. Therefore, `01:30` refers to two different instants.

Python represents the two possibilities with the `fold` attribute:

- `fold=0` selects the earlier occurrence,
- `fold=1` selects the later occurrence.

In [15]:
ambiguous_wall_time = datetime(2024, 11, 3, 1, 30)
ny_zone = ZoneInfo("America/New_York")

first_occurrence = ambiguous_wall_time.replace(
    tzinfo=ny_zone,
    fold=0,
)
second_occurrence = ambiguous_wall_time.replace(
    tzinfo=ny_zone,
    fold=1,
)

print(describe_datetime(first_occurrence))
print(describe_datetime(second_occurrence))

2024-11-03T01:30:00-04:00 | zone=EDT | UTC=2024-11-03T05:30:00+00:00
2024-11-03T01:30:00-05:00 | zone=EST | UTC=2024-11-03T06:30:00+00:00


### Step 1 — Compare the offsets

The wall-clock fields are identical, but the offsets differ.

In [16]:
print("First offset: ", first_occurrence.utcoffset())
print("Second offset:", second_occurrence.utcoffset())

First offset:  -1 day, 20:00:00
Second offset: -1 day, 19:00:00


### Step 2 — Compare the UTC instants

The two local datetimes are one hour apart on the UTC timeline.

In [17]:
first_utc = first_occurrence.astimezone(timezone.utc)
second_utc = second_occurrence.astimezone(timezone.utc)

print("First UTC: ", first_utc)
print("Second UTC:", second_utc)
print("Difference:", second_utc - first_utc)

First UTC:  2024-11-03 05:30:00+00:00
Second UTC: 2024-11-03 06:30:00+00:00
Difference: 1:00:00


### Step 3 — Build an explicit resolver

A production system should not silently guess when an input is ambiguous.

The following function requires the caller to choose `"earlier"` or `"later"`.

In [18]:
def resolve_ambiguous_time(
    local_time: datetime,
    zone_name: str,
    choice: str,
) -> datetime:
    if local_time.tzinfo is not None:
        raise ValueError("Pass a naive local wall-clock datetime.")

    if choice not in {"earlier", "later"}:
        raise ValueError("choice must be 'earlier' or 'later'.")

    zone = ZoneInfo(zone_name)
    fold = 0 if choice == "earlier" else 1

    candidate = local_time.replace(tzinfo=zone, fold=fold)
    other = local_time.replace(tzinfo=zone, fold=1 - fold)

    if candidate.utcoffset() == other.utcoffset():
        raise ValueError("The supplied local time is not ambiguous.")

    return candidate

In [19]:
earlier = resolve_ambiguous_time(
    datetime(2024, 11, 3, 1, 30),
    "America/New_York",
    "earlier",
)
later = resolve_ambiguous_time(
    datetime(2024, 11, 3, 1, 30),
    "America/New_York",
    "later",
)

print(earlier.astimezone(timezone.utc))
print(later.astimezone(timezone.utc))

2024-11-03 05:30:00+00:00
2024-11-03 06:30:00+00:00


### Solution verification

In [20]:
assert earlier.fold == 0
assert later.fold == 1
assert later.astimezone(timezone.utc) - earlier.astimezone(timezone.utc) == timedelta(hours=1)

print("Problem 3 checks passed.")

Problem 3 checks passed.


### Key idea

An ambiguous local time cannot be resolved from its clock fields alone.

Store or request enough information to choose the intended occurrence.

## Problem 4 — Classify local times as unique, ambiguous, or nonexistent

The spring daylight-saving transition creates the opposite problem: some local wall times never occur.

In New York on 10 March 2024, the clock jumped from `01:59:59` to `03:00:00`. Therefore, a value such as `02:30` is nonexistent.

`ZoneInfo` does not raise an exception merely because we attach a zone to a nonexistent wall time.

We need an explicit validation strategy.

In [21]:
nonexistent_wall_time = datetime(2024, 3, 10, 2, 30)
candidate = nonexistent_wall_time.replace(
    tzinfo=ZoneInfo("America/New_York")
)

print(describe_datetime(candidate))

2024-03-10T02:30:00-05:00 | zone=EST | UTC=2024-03-10T07:30:00+00:00


### Step 1 — Use a UTC round trip

For a valid local datetime:

1. attach the zone,
2. convert to UTC,
3. convert back to the original zone.

The final wall-clock fields should match the original input.

In [22]:
def survives_utc_round_trip(
    local_time: datetime,
    zone: ZoneInfo,
    fold: int,
) -> bool:
    candidate = local_time.replace(tzinfo=zone, fold=fold)
    returned = (
        candidate
        .astimezone(timezone.utc)
        .astimezone(zone)
        .replace(tzinfo=None)
    )
    return returned == local_time

In [23]:
ny_zone = ZoneInfo("America/New_York")

print(
    "fold=0 survives:",
    survives_utc_round_trip(nonexistent_wall_time, ny_zone, 0),
)
print(
    "fold=1 survives:",
    survives_utc_round_trip(nonexistent_wall_time, ny_zone, 1),
)

fold=0 survives: False
fold=1 survives: False


### Step 2 — Combine the round-trip test with offset comparison

- If neither fold survives, the time is nonexistent.
- If both survive and the offsets differ, the time is ambiguous.
- Otherwise, the time is unique.

In [24]:
def classify_local_time(
    local_time: datetime,
    zone_name: str,
) -> str:
    if local_time.tzinfo is not None:
        raise ValueError("Pass a naive local datetime.")

    zone = ZoneInfo(zone_name)

    first = local_time.replace(tzinfo=zone, fold=0)
    second = local_time.replace(tzinfo=zone, fold=1)

    first_valid = survives_utc_round_trip(local_time, zone, 0)
    second_valid = survives_utc_round_trip(local_time, zone, 1)

    if not first_valid and not second_valid:
        return "nonexistent"

    if (
        first_valid
        and second_valid
        and first.utcoffset() != second.utcoffset()
    ):
        return "ambiguous"

    return "unique"

### Step 3 — Test all three categories

In [25]:
classification_examples = [
    (
        datetime(2024, 3, 10, 2, 30),
        "America/New_York",
    ),
    (
        datetime(2024, 11, 3, 1, 30),
        "America/New_York",
    ),
    (
        datetime(2024, 2, 15, 12, 0),
        "America/New_York",
    ),
]

for local_time, zone_name in classification_examples:
    print(
        local_time,
        zone_name,
        "->",
        classify_local_time(local_time, zone_name),
    )

2024-03-10 02:30:00 America/New_York -> nonexistent
2024-11-03 01:30:00 America/New_York -> ambiguous
2024-02-15 12:00:00 America/New_York -> unique


### Solution verification

In [26]:
assert classify_local_time(
    datetime(2024, 3, 10, 2, 30),
    "America/New_York",
) == "nonexistent"

assert classify_local_time(
    datetime(2024, 11, 3, 1, 30),
    "America/New_York",
) == "ambiguous"

assert classify_local_time(
    datetime(2024, 2, 15, 12, 0),
    "America/New_York",
) == "unique"

print("Problem 4 checks passed.")

Problem 4 checks passed.


### Key idea

Attaching `tzinfo` is not, by itself, validation.

For user-entered local times near DST boundaries, classify the input before accepting it.

## Problem 5 — Find a common meeting window across offices

Three offices publish local availability windows for the same calendar date.

We need to find the overlap on the UTC timeline and then display the result in every participant's local zone.

In [27]:
office_windows = [
    {
        "office": "Sofia",
        "zone": "Europe/Sofia",
        "date": date(2025, 2, 12),
        "start": time(15, 0),
        "end": time(18, 0),
    },
    {
        "office": "New York",
        "zone": "America/New_York",
        "date": date(2025, 2, 12),
        "start": time(8, 0),
        "end": time(12, 0),
    },
    {
        "office": "London",
        "zone": "Europe/London",
        "date": date(2025, 2, 12),
        "start": time(13, 30),
        "end": time(16, 30),
    },
]

office_windows

[{'office': 'Sofia',
  'zone': 'Europe/Sofia',
  'date': datetime.date(2025, 2, 12),
  'start': datetime.time(15, 0),
  'end': datetime.time(18, 0)},
 {'office': 'New York',
  'zone': 'America/New_York',
  'date': datetime.date(2025, 2, 12),
  'start': datetime.time(8, 0),
  'end': datetime.time(12, 0)},
 {'office': 'London',
  'zone': 'Europe/London',
  'date': datetime.date(2025, 2, 12),
  'start': datetime.time(13, 30),
  'end': datetime.time(16, 30)}]

### Step 1 — Convert one local interval to UTC

In [28]:
window = office_windows[0]
zone = ZoneInfo(window["zone"])

local_start = datetime.combine(
    window["date"],
    window["start"],
    tzinfo=zone,
)
local_end = datetime.combine(
    window["date"],
    window["end"],
    tzinfo=zone,
)

print("Local:", local_start, "to", local_end)
print(
    "UTC:  ",
    local_start.astimezone(timezone.utc),
    "to",
    local_end.astimezone(timezone.utc),
)

Local: 2025-02-12 15:00:00+02:00 to 2025-02-12 18:00:00+02:00
UTC:   2025-02-12 13:00:00+00:00 to 2025-02-12 16:00:00+00:00


### Step 2 — Normalize all intervals

For a set of intervals, the common overlap begins at the latest start and ends at the earliest end.

In [29]:
def normalize_window(window: Mapping[str, Any]) -> dict[str, Any]:
    zone = ZoneInfo(window["zone"])

    local_start = datetime.combine(
        window["date"],
        window["start"],
        tzinfo=zone,
    )
    local_end = datetime.combine(
        window["date"],
        window["end"],
        tzinfo=zone,
    )

    if local_end <= local_start:
        raise ValueError("Each window must end after it starts.")

    return {
        **window,
        "local_start": local_start,
        "local_end": local_end,
        "utc_start": local_start.astimezone(timezone.utc),
        "utc_end": local_end.astimezone(timezone.utc),
    }


normalized_windows = [
    normalize_window(window)
    for window in office_windows
]

for window in normalized_windows:
    print(
        window["office"],
        window["utc_start"].strftime("%H:%M"),
        "-",
        window["utc_end"].strftime("%H:%M"),
        "UTC",
    )

Sofia 13:00 - 16:00 UTC
New York 13:00 - 17:00 UTC
London 13:30 - 16:30 UTC


### Step 3 — Compute the intersection

In [30]:
def intersect_windows(
    windows: Iterable[Mapping[str, Any]],
) -> tuple[datetime, datetime] | None:
    normalized = [normalize_window(window) for window in windows]

    if not normalized:
        return None

    overlap_start = max(
        window["utc_start"]
        for window in normalized
    )
    overlap_end = min(
        window["utc_end"]
        for window in normalized
    )

    if overlap_start >= overlap_end:
        return None

    return overlap_start, overlap_end

In [31]:
overlap = intersect_windows(office_windows)
overlap

(datetime.datetime(2025, 2, 12, 13, 30, tzinfo=datetime.timezone.utc),
 datetime.datetime(2025, 2, 12, 16, 0, tzinfo=datetime.timezone.utc))

### Step 4 — Display the overlap locally

In [32]:
assert overlap is not None
overlap_start, overlap_end = overlap

for window in office_windows:
    zone = ZoneInfo(window["zone"])
    print(
        window["office"],
        ":",
        overlap_start.astimezone(zone).strftime("%H:%M %Z"),
        "-",
        overlap_end.astimezone(zone).strftime("%H:%M %Z"),
    )

Sofia : 15:30 EET - 18:00 EET
New York : 08:30 EST - 11:00 EST
London : 13:30 GMT - 16:00 GMT


### Solution verification

In [33]:
assert overlap_start == datetime(
    2025, 2, 12, 13, 30, tzinfo=timezone.utc
)
assert overlap_end == datetime(
    2025, 2, 12, 16, 0, tzinfo=timezone.utc
)

print("Problem 5 checks passed.")

Problem 5 checks passed.


### Key idea

Intervals from different locations should be intersected on one common timeline, normally UTC.

Convert only after the local business rule has been applied.

# Part 2 — Advanced `math.gcd()` and `math.lcm()` Problems

Python 3.9 allows `math.gcd()` to accept multiple arguments and introduces `math.lcm()`.

These operations are especially useful for normalization, batching, tiling, and periodic schedules.

## Problem 6 — Normalize a multi-part ratio

A media profile is described by three integer measurements:

- width,
- height,
- frame rate.

We want the simplest whole-number ratio that preserves all three proportions.

In [34]:
profile = (3840, 2160, 120)
profile

(3840, 2160, 120)

### Step 1 — Find the common divisor

The largest integer that divides every component is the multi-argument GCD.

In [35]:
common_divisor = gcd(*profile)
common_divisor

120

### Step 2 — Divide every component

In [36]:
normalized_profile = tuple(
    value // common_divisor
    for value in profile
)

normalized_profile

(32, 18, 1)

### Step 3 — Generalize the operation

We also need to decide what should happen when:

- a value is negative,
- some values are zero,
- every value is zero.

In [37]:
def normalize_integer_ratio(
    values: Iterable[int],
) -> tuple[int, ...]:
    values = tuple(values)

    if not values:
        raise ValueError("At least one value is required.")

    divisor = gcd(*(abs(value) for value in values))

    if divisor == 0:
        raise ValueError("An all-zero ratio cannot be normalized.")

    return tuple(value // divisor for value in values)

In [38]:
ratio_examples = [
    (3840, 2160, 120),
    (150, 225, 375),
    (-24, 36, 60),
    (0, 18, 30),
]

for values in ratio_examples:
    print(values, "->", normalize_integer_ratio(values))

(3840, 2160, 120) -> (32, 18, 1)
(150, 225, 375) -> (2, 3, 5)
(-24, 36, 60) -> (-2, 3, 5)
(0, 18, 30) -> (0, 3, 5)


### Solution verification

In [39]:
assert normalize_integer_ratio((3840, 2160, 120)) == (32, 18, 1)
assert normalize_integer_ratio((150, 225, 375)) == (2, 3, 5)
assert normalize_integer_ratio((0, 18, 30)) == (0, 3, 5)

print("Problem 6 checks passed.")

Problem 6 checks passed.


### Key idea

A multi-argument GCD reduces an entire integer vector in one operation.

Normalize the signs only when the domain requires a particular sign convention.

## Problem 7 — Choose the largest exact batch size

A warehouse must split several product quantities into equal-sized batches.

No product may have a partial batch. We want the largest possible common batch size.

In [40]:
quantities = {
    "sensors": 840,
    "cables": 630,
    "controllers": 1050,
    "mounts": 420,
}

quantities

{'sensors': 840, 'cables': 630, 'controllers': 1050, 'mounts': 420}

### Step 1 — Compute the largest shared batch size

In [41]:
batch_size = gcd(*quantities.values())
batch_size

210

### Step 2 — Compute the number of batches for each product

In [42]:
batch_plan = {
    product: quantity // batch_size
    for product, quantity in quantities.items()
}

batch_plan

{'sensors': 4, 'cables': 3, 'controllers': 5, 'mounts': 2}

### Step 3 — Package the result in a validation-friendly function

In [43]:
def make_exact_batch_plan(
    quantities: Mapping[str, int],
) -> dict[str, Any]:
    if not quantities:
        raise ValueError("At least one quantity is required.")

    if any(quantity <= 0 for quantity in quantities.values()):
        raise ValueError("All quantities must be positive.")

    size = gcd(*quantities.values())

    counts = {
        item: quantity // size
        for item, quantity in quantities.items()
    }

    return {
        "batch_size": size,
        "batch_counts": counts,
    }

In [44]:
warehouse_plan = make_exact_batch_plan(quantities)
warehouse_plan

{'batch_size': 210,
 'batch_counts': {'sensors': 4, 'cables': 3, 'controllers': 5, 'mounts': 2}}

### Solution verification

In [45]:
assert warehouse_plan["batch_size"] == 210

for product, quantity in quantities.items():
    count = warehouse_plan["batch_counts"][product]
    assert count * warehouse_plan["batch_size"] == quantity

print("Problem 7 checks passed.")

Problem 7 checks passed.


### Key idea

Use GCD when the question asks for the largest unit that divides every quantity exactly.

## Problem 8 — Find when periodic jobs align

Three maintenance jobs repeat every:

- 12 minutes,
- 18 minutes,
- 30 minutes.

They all start together at minute zero. We need to find the next time they all start together.

In [46]:
periods = (12, 18, 30)
alignment_minutes = lcm(*periods)
alignment_minutes

180

### Step 1 — Confirm the LCM manually by divisibility

In [47]:
for period in periods:
    print(
        alignment_minutes,
        "is divisible by",
        period,
        "->",
        alignment_minutes % period == 0,
    )

180 is divisible by 12 -> True
180 is divisible by 18 -> True
180 is divisible by 30 -> True


### Step 2 — Generate every alignment in an eight-hour shift

In [48]:
shift_minutes = 8 * 60

alignment_points = list(
    range(0, shift_minutes + 1, alignment_minutes)
)

alignment_points

[0, 180, 360]

### Step 3 — Convert the offsets into clock times

In [49]:
shift_start = datetime(
    2025, 2, 12, 8, 0,
    tzinfo=ZoneInfo("Europe/Sofia"),
)

alignment_times = [
    shift_start + timedelta(minutes=offset)
    for offset in alignment_points
]

for value in alignment_times:
    print(value.strftime("%H:%M %Z"))

08:00 EET
11:00 EET
14:00 EET


### Step 4 — Generalize the scheduler

In [50]:
def synchronized_offsets(
    periods: Iterable[int],
    horizon: int,
) -> list[int]:
    periods = tuple(periods)

    if not periods:
        raise ValueError("At least one period is required.")

    if any(period <= 0 for period in periods):
        raise ValueError("Periods must be positive.")

    if horizon < 0:
        raise ValueError("The horizon cannot be negative.")

    step = lcm(*periods)
    return list(range(0, horizon + 1, step))

### Solution verification

In [51]:
assert alignment_minutes == 180
assert synchronized_offsets((12, 18, 30), 480) == [0, 180, 360]

print("Problem 8 checks passed.")

Problem 8 checks passed.


### Key idea

Use LCM when the question asks for the earliest positive point that is a multiple of every period.

## Problem 9 — Tile several rectangular panels with one square size

A workshop has panels with dimensions measured in millimetres.

Every panel must be tiled with identical square tiles, with no cutting and no gaps. Find the largest possible tile side.

In [52]:
panels = [
    (840, 630),
    (1260, 945),
    (420, 315),
]

panels

[(840, 630), (1260, 945), (420, 315)]

### Step 1 — Flatten all dimensions

The square side must divide every width and every height.

In [53]:
all_dimensions = [
    dimension
    for panel in panels
    for dimension in panel
]

all_dimensions

[840, 630, 1260, 945, 420, 315]

### Step 2 — Find the shared divisor

In [54]:
tile_side = gcd(*all_dimensions)
tile_side

105

### Step 3 — Count the tiles needed for each panel

In [55]:
tile_counts = []

for width, height in panels:
    across = width // tile_side
    down = height // tile_side
    tile_counts.append(across * down)

tile_counts

[48, 108, 12]

### Step 4 — Produce a detailed report

In [56]:
def square_tile_report(
    panels: Iterable[tuple[int, int]],
) -> dict[str, Any]:
    panels = tuple(panels)

    if not panels:
        raise ValueError("At least one panel is required.")

    if any(
        width <= 0 or height <= 0
        for width, height in panels
    ):
        raise ValueError("Panel dimensions must be positive.")

    side = gcd(
        *[
            dimension
            for panel in panels
            for dimension in panel
        ]
    )

    details = []
    for width, height in panels:
        across = width // side
        down = height // side
        details.append(
            {
                "panel": (width, height),
                "tiles_across": across,
                "tiles_down": down,
                "tile_count": across * down,
            }
        )

    return {
        "tile_side": side,
        "details": details,
        "total_tiles": sum(
            detail["tile_count"]
            for detail in details
        ),
    }

In [57]:
tiling = square_tile_report(panels)
tiling

{'tile_side': 105,
 'details': [{'panel': (840, 630),
   'tiles_across': 8,
   'tiles_down': 6,
   'tile_count': 48},
  {'panel': (1260, 945),
   'tiles_across': 12,
   'tiles_down': 9,
   'tile_count': 108},
  {'panel': (420, 315), 'tiles_across': 4, 'tiles_down': 3, 'tile_count': 12}],
 'total_tiles': 168}

### Solution verification

In [58]:
assert tiling["tile_side"] == 105
assert tiling["total_tiles"] == sum(tile_counts)

for detail in tiling["details"]:
    width, height = detail["panel"]
    assert width % tiling["tile_side"] == 0
    assert height % tiling["tile_side"] == 0

print("Problem 9 checks passed.")

Problem 9 checks passed.


### Key idea

A geometric tiling problem often becomes a GCD problem after all relevant dimensions are collected.

## Problem 10 — Find the greatest common time quantum for fractional durations

Three processing stages take:

- `3/4` second,
- `5/6` second,
- `7/10` second.

We want the largest rational time quantum that divides every duration exactly.

In [59]:
durations = (
    Fraction(3, 4),
    Fraction(5, 6),
    Fraction(7, 10),
)

durations

(Fraction(3, 4), Fraction(5, 6), Fraction(7, 10))

### Step 1 — Move all fractions to a common denominator

The least common multiple of the denominators gives a denominator that can represent every duration exactly.

In [60]:
common_denominator = lcm(
    *(duration.denominator for duration in durations)
)
common_denominator

60

### Step 2 — Convert the durations into integer tick counts

In [61]:
tick_counts = [
    duration.numerator
    * (common_denominator // duration.denominator)
    for duration in durations
]

tick_counts

[45, 50, 42]

### Step 3 — Find the greatest common divisor of the tick counts

In [62]:
common_tick_count = gcd(*tick_counts)
common_quantum = Fraction(
    common_tick_count,
    common_denominator,
)

common_quantum

Fraction(1, 60)

### Step 4 — Generalize the algorithm

In [63]:
def fraction_gcd(
    values: Iterable[Fraction],
) -> Fraction:
    values = tuple(Fraction(value) for value in values)

    if not values:
        raise ValueError("At least one fraction is required.")

    if any(value <= 0 for value in values):
        raise ValueError("All durations must be positive.")

    denominator = lcm(
        *(value.denominator for value in values)
    )

    integer_counts = [
        value.numerator
        * (denominator // value.denominator)
        for value in values
    ]

    return Fraction(gcd(*integer_counts), denominator)

In [64]:
quantum = fraction_gcd(durations)

print("Greatest common quantum:", quantum, "seconds")
print(
    "Stage lengths in quanta:",
    [duration / quantum for duration in durations],
)

Greatest common quantum: 1/60 seconds
Stage lengths in quanta: [Fraction(45, 1), Fraction(50, 1), Fraction(42, 1)]


### Solution verification

In [65]:
assert quantum == Fraction(1, 60)
assert all(
    (duration / quantum).denominator == 1
    for duration in durations
)

print("Problem 10 checks passed.")

Problem 10 checks passed.


### Key idea

For rational values:

1. use LCM to build a common denominator,
2. convert to integers,
3. use GCD on the integer counts.

# Part 3 — Advanced Dictionary Union Problems

For dictionaries:

```python
left | right
```

creates a new dictionary.

When the same key exists in both operands, the value from the right operand wins.

The original key position is preserved for overwritten keys, while genuinely new keys are appended.

## Problem 11 — Build a layered configuration

An application has four configuration layers:

1. built-in defaults,
2. a deployment file,
3. environment-specific overrides,
4. command-line overrides.

Later layers must take precedence.

In [66]:
defaults = {
    "host": "127.0.0.1",
    "port": 8000,
    "debug": False,
    "log_level": "INFO",
}

deployment = {
    "host": "0.0.0.0",
    "workers": 4,
}

environment = {
    "port": 8080,
    "log_level": "WARNING",
}

command_line = {
    "debug": True,
}

### Step 1 — Merge two layers

The right-hand dictionary wins on conflicts.

In [67]:
defaults | deployment

{'host': '0.0.0.0',
 'port': 8000,
 'debug': False,
 'log_level': 'INFO',
 'workers': 4}

### Step 2 — Merge the complete precedence chain

In [68]:
effective_config = (
    defaults
    | deployment
    | environment
    | command_line
)

effective_config

{'host': '0.0.0.0',
 'port': 8080,
 'debug': True,
 'log_level': 'WARNING',
 'workers': 4}

### Step 3 — Confirm that the inputs were not mutated

In [69]:
print("Defaults:", defaults)
print("Deployment:", deployment)
print("Environment:", environment)
print("Command line:", command_line)

Defaults: {'host': '127.0.0.1', 'port': 8000, 'debug': False, 'log_level': 'INFO'}
Deployment: {'host': '0.0.0.0', 'workers': 4}
Environment: {'port': 8080, 'log_level': 'WARNING'}
Command line: {'debug': True}


### Step 4 — Generalize the operation

In [70]:
def merge_layers(
    *layers: Mapping[str, Any],
) -> dict[str, Any]:
    result: dict[str, Any] = {}

    for layer in layers:
        result |= dict(layer)

    return result

In [71]:
config_from_function = merge_layers(
    defaults,
    deployment,
    environment,
    command_line,
)

config_from_function

{'host': '0.0.0.0',
 'port': 8080,
 'debug': True,
 'log_level': 'WARNING',
 'workers': 4}

### Solution verification

In [72]:
assert config_from_function == effective_config
assert effective_config["host"] == "0.0.0.0"
assert effective_config["port"] == 8080
assert effective_config["debug"] is True
assert effective_config["workers"] == 4

print("Problem 11 checks passed.")

Problem 11 checks passed.


### Key idea

Write configuration layers from lowest precedence to highest precedence.

The reading order then matches the override rule.

## Problem 12 — Report conflicts and value provenance

A plain union gives the effective value, but it does not tell us where that value came from.

We need both:

- the merged result,
- a provenance map showing the winning layer for every key.

We will retain the same precedence rule as the previous problem: later layers win.

In [73]:
named_layers = [
    ("defaults", defaults),
    ("deployment", deployment),
    ("environment", environment),
    ("command_line", command_line),
]

### Step 1 — Track the winner while merging

In [74]:
def merge_with_provenance(
    named_layers: Iterable[
        tuple[str, Mapping[str, Any]]
    ],
) -> tuple[dict[str, Any], dict[str, str]]:
    merged: dict[str, Any] = {}
    provenance: dict[str, str] = {}

    for layer_name, layer in named_layers:
        merged |= dict(layer)

        for key in layer:
            provenance[key] = layer_name

    return merged, provenance

In [75]:
merged_config, provenance = merge_with_provenance(
    named_layers
)

print("Merged:", merged_config)
print("Sources:", provenance)

Merged: {'host': '0.0.0.0', 'port': 8080, 'debug': True, 'log_level': 'WARNING', 'workers': 4}
Sources: {'host': 'deployment', 'port': 'environment', 'debug': 'command_line', 'log_level': 'environment', 'workers': 'deployment'}


### Step 2 — Build a conflict history

For auditing, we also want every overwritten value, not only the final winner.

In [76]:
def merge_with_history(
    named_layers: Iterable[
        tuple[str, Mapping[str, Any]]
    ],
) -> tuple[
    dict[str, Any],
    dict[str, list[tuple[str, Any]]],
]:
    merged: dict[str, Any] = {}
    history: dict[str, list[tuple[str, Any]]] = {}

    for layer_name, layer in named_layers:
        for key, value in layer.items():
            history.setdefault(key, []).append(
                (layer_name, value)
            )

        merged |= dict(layer)

    return merged, history

In [77]:
merged_config, history = merge_with_history(named_layers)

for key, entries in history.items():
    if len(entries) > 1:
        print(key, "was defined by", entries)

host was defined by [('defaults', '127.0.0.1'), ('deployment', '0.0.0.0')]
port was defined by [('defaults', 8000), ('environment', 8080)]
debug was defined by [('defaults', False), ('command_line', True)]
log_level was defined by [('defaults', 'INFO'), ('environment', 'WARNING')]


### Solution verification

In [78]:
assert provenance["port"] == "environment"
assert provenance["debug"] == "command_line"
assert history["host"] == [
    ("defaults", "127.0.0.1"),
    ("deployment", "0.0.0.0"),
]

print("Problem 12 checks passed.")

Problem 12 checks passed.


### Key idea

Dictionary union solves precedence.

If the application also needs explanation or auditability, collect provenance during the same left-to-right pass.

## Problem 13 — Demonstrate that dictionary union is shallow

Consider two nested configuration dictionaries.

A top-level union does not recursively merge nested dictionaries. The complete nested value from the right operand replaces the nested value from the left.

In [79]:
base_service = {
    "service": {
        "host": "api.internal",
        "port": 443,
        "tls": {
            "enabled": True,
            "verify": True,
        },
    },
    "retries": 3,
}

service_override = {
    "service": {
        "port": 8443,
        "tls": {
            "verify": False,
        },
    },
}

shallow_result = base_service | service_override
shallow_result

{'service': {'port': 8443, 'tls': {'verify': False}}, 'retries': 3}

The `host` field disappeared because the entire value associated with the top-level key `"service"` was replaced.

### Step 1 — Define the desired recursive rule

For each key:

- if both values are mappings, merge them recursively,
- otherwise, use the right-hand value.

In [80]:
def deep_merge(
    left: Mapping[str, Any],
    right: Mapping[str, Any],
) -> dict[str, Any]:
    result = dict(left)

    for key, right_value in right.items():
        left_value = result.get(key)

        if (
            isinstance(left_value, Mapping)
            and isinstance(right_value, Mapping)
        ):
            result[key] = deep_merge(
                left_value,
                right_value,
            )
        else:
            result[key] = right_value

    return result

### Step 2 — Apply the recursive merge

In [81]:
deep_result = deep_merge(
    base_service,
    service_override,
)

deep_result

{'service': {'host': 'api.internal',
  'port': 8443,
  'tls': {'enabled': True, 'verify': False}},
 'retries': 3}

### Step 3 — Compare the shallow and deep results

In [82]:
print("Shallow service:", shallow_result["service"])
print("Deep service:   ", deep_result["service"])

Shallow service: {'port': 8443, 'tls': {'verify': False}}
Deep service:    {'host': 'api.internal', 'port': 8443, 'tls': {'enabled': True, 'verify': False}}


### Solution verification

In [83]:
assert "host" not in shallow_result["service"]
assert deep_result["service"]["host"] == "api.internal"
assert deep_result["service"]["port"] == 8443
assert deep_result["service"]["tls"] == {
    "enabled": True,
    "verify": False,
}

print("Problem 13 checks passed.")

Problem 13 checks passed.


### Key idea

The `|` operator is intentionally shallow.

Recursive merging requires an explicit domain-specific policy, especially for lists, sets, and incompatible value types.

## Problem 14 — Understand mutation with `|=`

The in-place union operator updates the dictionary on its left.

This is useful for an accumulator, but dangerous if other parts of the program expect the original dictionary to remain unchanged.

In [84]:
original_settings = {
    "timeout": 30,
    "retries": 2,
}

working_settings = original_settings
working_settings |= {
    "timeout": 45,
    "cache": True,
}

print("Original:", original_settings)
print("Working: ", working_settings)

Original: {'timeout': 45, 'retries': 2, 'cache': True}
Working:  {'timeout': 45, 'retries': 2, 'cache': True}


Both names refer to the same object, so both appear changed.

### Step 1 — Compare object identities

In [85]:
print(id(original_settings))
print(id(working_settings))
print(
    "Same object:",
    original_settings is working_settings,
)

2520142788032
2520142788032
Same object: True


### Step 2 — Copy before using `|=`

A shallow copy is enough when only top-level mutation is expected.

In [86]:
safe_original = {
    "timeout": 30,
    "retries": 2,
}

safe_working = safe_original.copy()
safe_working |= {
    "timeout": 45,
    "cache": True,
}

print("Safe original:", safe_original)
print("Safe working: ", safe_working)

Safe original: {'timeout': 30, 'retries': 2}
Safe working:  {'timeout': 45, 'retries': 2, 'cache': True}


### Step 3 — Encapsulate the intent

In [87]:
def updated_copy(
    base: Mapping[str, Any],
    *updates: Mapping[str, Any],
) -> dict[str, Any]:
    result = dict(base)

    for update in updates:
        result |= dict(update)

    return result

In [88]:
new_settings = updated_copy(
    safe_original,
    {"timeout": 60},
    {"cache": True},
)

new_settings

{'timeout': 60, 'retries': 2, 'cache': True}

### Solution verification

In [89]:
assert safe_original == {
    "timeout": 30,
    "retries": 2,
}
assert new_settings == {
    "timeout": 60,
    "retries": 2,
    "cache": True,
}

print("Problem 14 checks passed.")

Problem 14 checks passed.


### Key idea

Use `|` when you want a new dictionary.

Use `|=` when deliberate mutation is part of the design.

## Problem 15 — Merge records while preserving a stable display order

A dashboard begins with a stable set of metrics.

A later data source updates some existing values and adds new metrics. Existing keys should keep their original positions, while new keys should appear at the end.

In [90]:
baseline_metrics = {
    "requests": 1200,
    "errors": 18,
    "latency_ms": 84,
}

latest_metrics = {
    "errors": 11,
    "throughput": 250,
    "latency_ms": 79,
}

merged_metrics = baseline_metrics | latest_metrics
merged_metrics

{'requests': 1200, 'errors': 11, 'latency_ms': 79, 'throughput': 250}

The value of `"errors"` comes from the right dictionary, but the key remains in its original second position.

### Step 1 — Inspect the key order explicitly

In [91]:
list(merged_metrics)

['requests', 'errors', 'latency_ms', 'throughput']

### Step 2 — Build a change report without changing that order

In [92]:
def ordered_change_report(
    baseline: Mapping[str, Any],
    latest: Mapping[str, Any],
) -> list[dict[str, Any]]:
    merged = dict(baseline) | dict(latest)

    report = []
    for key, value in merged.items():
        if key not in baseline:
            status = "added"
            previous = None
        elif key in latest and latest[key] != baseline[key]:
            status = "changed"
            previous = baseline[key]
        else:
            status = "unchanged"
            previous = baseline[key]

        report.append(
            {
                "metric": key,
                "previous": previous,
                "current": value,
                "status": status,
            }
        )

    return report

In [93]:
metric_report = ordered_change_report(
    baseline_metrics,
    latest_metrics,
)

for row in metric_report:
    print(row)

{'metric': 'requests', 'previous': 1200, 'current': 1200, 'status': 'unchanged'}
{'metric': 'errors', 'previous': 18, 'current': 11, 'status': 'changed'}
{'metric': 'latency_ms', 'previous': 84, 'current': 79, 'status': 'changed'}
{'metric': 'throughput', 'previous': None, 'current': 250, 'status': 'added'}


### Solution verification

In [94]:
assert [row["metric"] for row in metric_report] == [
    "requests",
    "errors",
    "latency_ms",
    "throughput",
]
assert metric_report[1]["status"] == "changed"
assert metric_report[-1]["status"] == "added"

print("Problem 15 checks passed.")

Problem 15 checks passed.


### Key idea

An overwritten value does not move its key to the end.

This makes dictionary union useful when a stable presentation order matters.

# Part 4 — Advanced Prefix and Suffix Problems

`removeprefix()` and `removesuffix()` remove one exact affix.

They do not interpret the argument as a set of characters, and they do nothing when the exact affix is absent.

## Problem 16 — Explain why `strip()` is unsafe for exact markers

Suppose log lines may begin with the exact marker `"(log) "`.

A common mistake is to use `lstrip("(log) ")`, but `lstrip()` treats its argument as a set of removable characters.

In [95]:
line = "(log) log: operation completed"

print("lstrip result:      ", line.lstrip("(log) "))
print("removeprefix result:", line.removeprefix("(log) "))

lstrip result:       : operation completed
removeprefix result: log: operation completed


The `lstrip()` call repeatedly removes any leading character found in the set:

```text
( l o g ) space
```

It is not checking for one exact prefix.

### Step 1 — Compare several edge cases

In [96]:
log_lines = [
    "(log) log: operation completed",
    "(log) object loaded",
    "logical result",
    "ordinary line",
]

for value in log_lines:
    print(
        value,
        "->",
        value.removeprefix("(log) "),
    )

(log) log: operation completed -> log: operation completed
(log) object loaded -> object loaded
logical result -> logical result
ordinary line -> ordinary line


### Step 2 — Build a safe cleaner

In [97]:
def remove_exact_marker(
    lines: Iterable[str],
    marker: str,
) -> list[str]:
    if marker == "":
        raise ValueError("The marker cannot be empty.")

    return [
        line.removeprefix(marker)
        for line in lines
    ]

In [98]:
cleaned_logs = remove_exact_marker(
    log_lines,
    "(log) ",
)

cleaned_logs

['log: operation completed',
 'object loaded',
 'logical result',
 'ordinary line']

### Solution verification

In [99]:
assert cleaned_logs == [
    "log: operation completed",
    "object loaded",
    "logical result",
    "ordinary line",
]

print("Problem 16 checks passed.")

Problem 16 checks passed.


### Key idea

Use `removeprefix()` when the requirement says “remove this exact leading substring once.”

## Problem 17 — Remove a known chain of filename suffixes

A data pipeline receives files such as:

- `events.csv.gz`,
- `snapshot.json.gz`,
- `notes.txt`.

We want to remove only a known compression suffix and then only a known data-format suffix.

In [100]:
filenames = [
    "events.csv.gz",
    "snapshot.json.gz",
    "notes.txt",
    "archive.tar.gz",
    "report.csv",
]

filenames

['events.csv.gz',
 'snapshot.json.gz',
 'notes.txt',
 'archive.tar.gz',
 'report.csv']

### Step 1 — Remove the compression suffix once

In [101]:
without_compression = [
    name.removesuffix(".gz")
    for name in filenames
]

without_compression

['events.csv', 'snapshot.json', 'notes.txt', 'archive.tar', 'report.csv']

### Step 2 — Remove one recognized data suffix

Calling `removesuffix()` repeatedly is explicit and does not damage unrelated endings.

In [102]:
def remove_one_known_suffix(
    value: str,
    suffixes: Iterable[str],
) -> tuple[str, str | None]:
    for suffix in suffixes:
        if value.endswith(suffix):
            return value.removesuffix(suffix), suffix

    return value, None

In [103]:
for name in without_compression:
    stem, removed = remove_one_known_suffix(
        name,
        [".csv", ".json", ".txt"],
    )
    print(name, "->", stem, "| removed:", removed)

events.csv -> events | removed: .csv
snapshot.json -> snapshot | removed: .json
notes.txt -> notes | removed: .txt
archive.tar -> archive.tar | removed: None
report.csv -> report | removed: .csv


### Step 3 — Build a complete filename parser

In [104]:
def parse_data_filename(name: str) -> dict[str, Any]:
    without_compression = name.removesuffix(".gz")
    compressed = without_compression != name

    stem, data_suffix = remove_one_known_suffix(
        without_compression,
        [".csv", ".json", ".txt"],
    )

    return {
        "original": name,
        "stem": stem,
        "data_suffix": data_suffix,
        "compressed": compressed,
    }

In [105]:
parsed_files = [
    parse_data_filename(name)
    for name in filenames
]

for parsed in parsed_files:
    print(parsed)

{'original': 'events.csv.gz', 'stem': 'events', 'data_suffix': '.csv', 'compressed': True}
{'original': 'snapshot.json.gz', 'stem': 'snapshot', 'data_suffix': '.json', 'compressed': True}
{'original': 'notes.txt', 'stem': 'notes', 'data_suffix': '.txt', 'compressed': False}
{'original': 'archive.tar.gz', 'stem': 'archive.tar', 'data_suffix': None, 'compressed': True}
{'original': 'report.csv', 'stem': 'report', 'data_suffix': '.csv', 'compressed': False}


### Solution verification

In [106]:
assert parse_data_filename("events.csv.gz") == {
    "original": "events.csv.gz",
    "stem": "events",
    "data_suffix": ".csv",
    "compressed": True,
}
assert parse_data_filename("archive.tar.gz")["stem"] == "archive.tar"

print("Problem 17 checks passed.")

Problem 17 checks passed.


### Key idea

Remove known suffixes in a documented order.

Do not use `rstrip(".gz")`; it removes any trailing combination of `.`, `g`, and `z`.

## Problem 18 — Parse an exact wrapper around an identifier

An external system sends identifiers in the form:

```text
urn:job:<identifier>:active
```

We need to remove the exact prefix and exact suffix, while rejecting malformed values.

In [107]:
identifiers = [
    "urn:job:alpha-17:active",
    "urn:job:beta-04:active",
    "job:gamma:active",
    "urn:job:delta:disabled",
]

identifiers

['urn:job:alpha-17:active',
 'urn:job:beta-04:active',
 'job:gamma:active',
 'urn:job:delta:disabled']

### Step 1 — Observe the no-op behavior

When an affix is absent, `removeprefix()` or `removesuffix()` returns the original string.

That behavior is convenient for cleanup, but a strict parser must validate the format first.

In [108]:
for identifier in identifiers:
    print(
        identifier,
        "->",
        identifier
        .removeprefix("urn:job:")
        .removesuffix(":active"),
    )

urn:job:alpha-17:active -> alpha-17
urn:job:beta-04:active -> beta-04
job:gamma:active -> job:gamma
urn:job:delta:disabled -> delta:disabled


### Step 2 — Validate before removing

In [109]:
def parse_active_job_id(value: str) -> str:
    prefix = "urn:job:"
    suffix = ":active"

    if not value.startswith(prefix):
        raise ValueError(
            f"Missing required prefix {prefix!r}."
        )

    if not value.endswith(suffix):
        raise ValueError(
            f"Missing required suffix {suffix!r}."
        )

    core = (
        value
        .removeprefix(prefix)
        .removesuffix(suffix)
    )

    if not core:
        raise ValueError("The identifier body cannot be empty.")

    return core

In [110]:
valid_job_ids = [
    parse_active_job_id(value)
    for value in identifiers[:2]
]

valid_job_ids

['alpha-17', 'beta-04']

### Step 3 — Demonstrate controlled failures

In [111]:
for value in identifiers[2:]:
    try:
        parse_active_job_id(value)
    except ValueError as exc:
        print(value, "->", exc)

job:gamma:active -> Missing required prefix 'urn:job:'.
urn:job:delta:disabled -> Missing required suffix ':active'.


### Solution verification

In [112]:
assert valid_job_ids == ["alpha-17", "beta-04"]

print("Problem 18 checks passed.")

Problem 18 checks passed.


### Key idea

For permissive cleanup, the no-op behavior is useful.

For strict parsing, validate the required format before removing the affixes.

## Problem 19 — Build an idempotent line normalizer

An idempotent operation gives the same result when applied repeatedly:

```python
normalize(normalize(x)) == normalize(x)
```

This property is useful in cleaning pipelines because accidental repeated execution does not continue changing the data.

In [113]:
raw_messages = [
    "[INFO] Service started\n",
    "[WARN] Cache nearing limit\n",
    "Service stopped\n",
]

### Step 1 — Define one exact cleanup pass

We will remove:

- one trailing newline,
- one recognized level prefix.

In [114]:
def normalize_message(value: str) -> str:
    result = value.removesuffix("\n")

    for prefix in ("[INFO] ", "[WARN] ", "[ERROR] "):
        if result.startswith(prefix):
            result = result.removeprefix(prefix)
            break

    return result

In [115]:
normalized_messages = [
    normalize_message(value)
    for value in raw_messages
]

normalized_messages

['Service started', 'Cache nearing limit', 'Service stopped']

### Step 2 — Apply the function twice

In [116]:
normalized_twice = [
    normalize_message(value)
    for value in normalized_messages
]

normalized_twice

['Service started', 'Cache nearing limit', 'Service stopped']

### Step 3 — Verify idempotence over a larger sample

In [117]:
message_samples = [
    "[INFO] Ready\n",
    "[WARN] Slow",
    "[ERROR] Failed\n",
    "Plain message",
    "",
]

for value in message_samples:
    once = normalize_message(value)
    twice = normalize_message(once)
    print(repr(value), "->", repr(once))
    assert twice == once

'[INFO] Ready\n' -> 'Ready'
'[WARN] Slow' -> 'Slow'
'[ERROR] Failed\n' -> 'Failed'
'Plain message' -> 'Plain message'
'' -> ''


### Solution verification

In [118]:
assert normalized_messages == [
    "Service started",
    "Cache nearing limit",
    "Service stopped",
]

print("Problem 19 checks passed.")

Problem 19 checks passed.


### Key idea

Exact one-time prefix and suffix removal naturally supports idempotent cleanup when the cleaned result no longer contains the recognized marker.

# Part 5 — Capstone Problem

The final problem combines all four feature groups in one small data-processing workflow.

## Problem 20 — Build a global deployment plan

A deployment request contains:

- layered configuration,
- an artifact name with known wrappers,
- shard weights that must be normalized,
- recurring health-check periods that must be synchronized,
- a local deployment wall time and IANA time zone.

We will transform the request into one validated deployment manifest.

In [119]:
deployment_request = {
    "defaults": {
        "strategy": "rolling",
        "replicas": 3,
        "timeout_seconds": 30,
    },
    "environment": {
        "replicas": 6,
        "region": "eu-central",
    },
    "request_overrides": {
        "timeout_seconds": 45,
    },
    "artifact": "artifact://payments-api-2025.02.12.tar.gz",
    "shard_weights": (120, 180, 300),
    "health_periods_seconds": (12, 18, 30),
    "local_deployment_time": datetime(
        2025, 2, 12, 21, 30
    ),
    "zone": "Europe/Sofia",
}

deployment_request

{'defaults': {'strategy': 'rolling', 'replicas': 3, 'timeout_seconds': 30},
 'environment': {'replicas': 6, 'region': 'eu-central'},
 'request_overrides': {'timeout_seconds': 45},
 'artifact': 'artifact://payments-api-2025.02.12.tar.gz',
 'shard_weights': (120, 180, 300),
 'health_periods_seconds': (12, 18, 30),
 'local_deployment_time': datetime.datetime(2025, 2, 12, 21, 30),
 'zone': 'Europe/Sofia'}

### Step 1 — Merge the configuration layers

The request override has the highest precedence.

In [120]:
effective_deployment_config = (
    deployment_request["defaults"]
    | deployment_request["environment"]
    | deployment_request["request_overrides"]
)

effective_deployment_config

{'strategy': 'rolling',
 'replicas': 6,
 'timeout_seconds': 45,
 'region': 'eu-central'}

### Step 2 — Clean the artifact name

We remove the exact transport prefix and then the known archive suffix.

In [121]:
clean_artifact = (
    deployment_request["artifact"]
    .removeprefix("artifact://")
    .removesuffix(".tar.gz")
)

clean_artifact

'payments-api-2025.02.12'

### Step 3 — Normalize shard weights

In [122]:
normalized_shard_weights = normalize_integer_ratio(
    deployment_request["shard_weights"]
)

normalized_shard_weights

(2, 3, 5)

### Step 4 — Synchronize the health checks

In [123]:
health_alignment_seconds = lcm(
    *deployment_request["health_periods_seconds"]
)

health_alignment_seconds

180

### Step 5 — Validate and convert the local deployment time

The request must not silently contain an ambiguous or nonexistent local time.

In [124]:
local_deployment_time = deployment_request[
    "local_deployment_time"
]
deployment_zone_name = deployment_request["zone"]

classification = classify_local_time(
    local_deployment_time,
    deployment_zone_name,
)

classification

'unique'

In [125]:
if classification != "unique":
    raise ValueError(
        "Deployment time must be unique; "
        f"received {classification!r}."
    )

deployment_local_aware = local_deployment_time.replace(
    tzinfo=ZoneInfo(deployment_zone_name)
)
deployment_utc = deployment_local_aware.astimezone(
    timezone.utc
)

print("Local:", deployment_local_aware)
print("UTC:  ", deployment_utc)

Local: 2025-02-12 21:30:00+02:00
UTC:   2025-02-12 19:30:00+00:00


### Step 6 — Assemble the final manifest

Dictionary union makes the final assembly readable while keeping the source request unchanged.

In [126]:
deployment_manifest = (
    effective_deployment_config
    | {
        "artifact": clean_artifact,
        "normalized_shard_weights": normalized_shard_weights,
        "health_alignment_seconds": health_alignment_seconds,
        "deployment_local": deployment_local_aware.isoformat(),
        "deployment_utc": deployment_utc.isoformat(),
        "deployment_zone": deployment_zone_name,
    }
)

deployment_manifest

{'strategy': 'rolling',
 'replicas': 6,
 'timeout_seconds': 45,
 'region': 'eu-central',
 'artifact': 'payments-api-2025.02.12',
 'normalized_shard_weights': (2, 3, 5),
 'health_alignment_seconds': 180,
 'deployment_local': '2025-02-12T21:30:00+02:00',
 'deployment_utc': '2025-02-12T19:30:00+00:00',
 'deployment_zone': 'Europe/Sofia'}

### Step 7 — Refactor the workflow into one function

In [127]:
def build_deployment_manifest(
    request: Mapping[str, Any],
) -> dict[str, Any]:
    config = merge_layers(
        request["defaults"],
        request["environment"],
        request["request_overrides"],
    )

    artifact = (
        request["artifact"]
        .removeprefix("artifact://")
        .removesuffix(".tar.gz")
    )

    shard_weights = normalize_integer_ratio(
        request["shard_weights"]
    )

    health_alignment = lcm(
        *request["health_periods_seconds"]
    )

    local_time = request["local_deployment_time"]
    zone_name = request["zone"]

    local_status = classify_local_time(
        local_time,
        zone_name,
    )

    if local_status != "unique":
        raise ValueError(
            "Deployment time must be unique; "
            f"received {local_status!r}."
        )

    local_aware = local_time.replace(
        tzinfo=ZoneInfo(zone_name)
    )
    utc_time = local_aware.astimezone(timezone.utc)

    return config | {
        "artifact": artifact,
        "normalized_shard_weights": shard_weights,
        "health_alignment_seconds": health_alignment,
        "deployment_local": local_aware.isoformat(),
        "deployment_utc": utc_time.isoformat(),
        "deployment_zone": zone_name,
    }

In [128]:
manifest_from_function = build_deployment_manifest(
    deployment_request
)

manifest_from_function

{'strategy': 'rolling',
 'replicas': 6,
 'timeout_seconds': 45,
 'region': 'eu-central',
 'artifact': 'payments-api-2025.02.12',
 'normalized_shard_weights': (2, 3, 5),
 'health_alignment_seconds': 180,
 'deployment_local': '2025-02-12T21:30:00+02:00',
 'deployment_utc': '2025-02-12T19:30:00+00:00',
 'deployment_zone': 'Europe/Sofia'}

### Capstone verification

In [129]:
assert manifest_from_function == deployment_manifest
assert manifest_from_function["replicas"] == 6
assert manifest_from_function["timeout_seconds"] == 45
assert manifest_from_function["artifact"] == "payments-api-2025.02.12"
assert manifest_from_function["normalized_shard_weights"] == (2, 3, 5)
assert manifest_from_function["health_alignment_seconds"] == 180
assert manifest_from_function["deployment_utc"].startswith(
    "2025-02-12T19:30:00+00:00"
)

print("Problem 20 checks passed.")

Problem 20 checks passed.


## Capstone design review

The workflow uses each feature for a specific reason:

- `|` expresses configuration precedence.
- `removeprefix()` and `removesuffix()` remove exact wrappers.
- `gcd()` reduces shard weights.
- `lcm()` synchronizes periodic tasks.
- `ZoneInfo` validates and converts local deployment time.

The result is deterministic, explicit, and testable.

# Additional Practice Problems with Compact Solutions

The following problems are shorter, but they extend the same ideas.

## Practice 1 — Convert one instant to several zones

Given a UTC instant and a list of IANA zone names, return a dictionary from zone name to localized datetime.

In [130]:
def convert_instant_to_zones(
    instant: datetime,
    zone_names: Iterable[str],
) -> dict[str, datetime]:
    if instant.tzinfo is None:
        raise ValueError("The instant must be timezone-aware.")

    utc_instant = instant.astimezone(timezone.utc)

    return {
        zone_name: utc_instant.astimezone(
            ZoneInfo(zone_name)
        )
        for zone_name in zone_names
    }


zone_conversion = convert_instant_to_zones(
    datetime(2025, 7, 1, 12, 0, tzinfo=timezone.utc),
    [
        "Europe/Sofia",
        "America/Los_Angeles",
        "Asia/Kolkata",
    ],
)

for zone_name, value in zone_conversion.items():
    print(zone_name, "->", value)

Europe/Sofia -> 2025-07-01 15:00:00+03:00
America/Los_Angeles -> 2025-07-01 05:00:00-07:00
Asia/Kolkata -> 2025-07-01 17:30:00+05:30


## Practice 2 — Find the smallest common packet capacity

Several packet types contain 8, 12, and 20 units. Find the smallest capacity that can hold a whole number of every packet type.

In [131]:
packet_sizes = (8, 12, 20)
common_capacity = lcm(*packet_sizes)

assert common_capacity == 120
common_capacity

120

## Practice 3 — Merge feature flags with environment overrides

The environment-specific values should win.

In [132]:
base_flags = {
    "search_v2": False,
    "new_checkout": False,
    "audit_log": True,
}

production_flags = {
    "search_v2": True,
    "new_checkout": True,
}

active_flags = base_flags | production_flags

assert active_flags == {
    "search_v2": True,
    "new_checkout": True,
    "audit_log": True,
}

active_flags

{'search_v2': True, 'new_checkout': True, 'audit_log': True}

## Practice 4 — Remove one exact Markdown wrapper

Remove one leading `"**"` and one trailing `"**"` only when they are both present.

In [133]:
def remove_bold_wrapper(value: str) -> str:
    if value.startswith("**") and value.endswith("**") and len(value) >= 4:
        return (
            value
            .removeprefix("**")
            .removesuffix("**")
        )

    return value


assert remove_bold_wrapper("**important**") == "important"
assert remove_bold_wrapper("important") == "important"
assert remove_bold_wrapper("**partial") == "**partial"

remove_bold_wrapper("**Python 3.9**")

'Python 3.9'

# Final Summary

The main problem-solving patterns are:

- **Time zones:** attach the correct zone, validate DST edge cases, normalize instants to UTC.
- **GCD:** find the largest exact shared unit.
- **LCM:** find the earliest shared multiple or synchronization point.
- **Dictionary union:** express shallow right-hand precedence clearly.
- **Prefix and suffix removal:** remove one exact affix without character-set behavior.

These features are small individually, but they become powerful when combined with validation and carefully defined business rules.